# A5: Content Creation at Scale

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from utils import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

## Creating Structured Output

In [2]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    platform: str = Field(..., description="The social media platform where the post will be published (e.g., Twitter, LinkedIn).")
    content: str = Field(..., description="The content of the social media post, including any hashtags or mentions.")

class ContentOutput(BaseModel):
    article: str = Field(..., description="The article, formatted in markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="A list of social media posts related to the article.")

## Loading Tasks and Agents YAML files

In [3]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/A5_agents.yaml',
    'tasks': 'config/A5_tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing CrewAI Tools

In [4]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

## Setup Multi LLM models

In [5]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

# Use OpenAI instead of Groq to avoid rate limits
# groq_llm = "groq/llama-3.3-70b-versatile"

# Using OpenAI's gpt-4o-mini model (already configured)
from langchain_openai import ChatOpenAI
openai_llm = ChatOpenAI(model='gpt-4o-mini')

## Creating Crew, Agents, and Tasks

In [6]:
# Creating Agents
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=openai_llm,
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
    llm=openai_llm,
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), WebsiteSearchTool()],
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
)

# Creating Tasks
monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

# Creating Crew
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
    ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

## Kicking off the Crew

In [7]:
result = content_creation_crew.kickoff(inputs={
  'subject': 'Inflation in the US and the impact on the stock market in 2024'
})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 122f472d-7a25-47db-a985-e53298d9104b                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Market Analyst                                                                                     │
│                                                                                                                 │
│  Task: Monitor and analyze the latest news and updates related to the financial markets, with a particular      │
│  focus on Inflation in the US and the impact on the stock market in 2024. Identify and summarize the most       │
│  relevant and impactful news items that could influence market trends or investor decisions. Utilize financial  │
│  news APIs and real-time market data tools to gather up-to-date information. Focus on detecting trends,         │
│  regulatory changes, or significant economic indicators that directly relate to Inflation in the US and the     │
│  impact on the stock market in 2024.                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Market Analyst                                                                                     │
│                                                                                                                 │
│  Thought: I need to gather the latest news and updates related to inflation in the U.S. and its impact on the   │
│  stock market in 2024.                                                                                          │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Content Officer                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "article": "# Understanding U.S. Inflation and Its Impact on the Stock Market in 2024\n\n## Introduction     │
│  \nAs we delve into 2024, inflation continues to be a significant factor influencing the U.S. economy and       │
│  stock market. After peaking at over 9% in June 2022, the current inflation rate stands at approximately 3.1%.  │
│  This blog post offers insights into how these inflationary trends are impacting investor sentiment, stock      │
│  market performance, and specific sectors.\n\n## Current Inflation Trends  \nInflation remains a pertinent      │
│  issue as the Federal Reserve explores potential interest rate cuts. As of December 2025, inflation in the      │
│  U.S. is approximately 3.1%, reflecting a moderate decline from previous peaks. Despite rising consumer         │
│  prices, the Consumer Price Index (CPI) indicates a more moderate inflationary environment compared to          │
│  previous years. Concerns about tariffs and their subsequent effect on consumer prices linger, contributing to  │
│  the current inflation dynamics.\n\nMoreover, the ongoing fluctuations in tariff policies contribute to         │
│  uncertainties in inflation forecasts. Consumers are becoming increasingly aware of the impact of these         │
│  tariffs, which could stifle purchasing power in the near future. This evolving situation necessitates          │
│  continuous monitoring from both investors and policymakers.\n\n## Impact on the Stock Market  \nThe stock      │
│  market is experiencing notable volatility, reflecting a divergence across various sectors influenced by        │
│  inflation. While sectors like technology and defense are thriving due to increased spending and digital        │
│  advancements, consumer staples are facing challenges from persistent inflation. Notably, the S&P 500 achieved  │
│  all-time highs throughout 2024, buoyed by resilient economic growth and corporate earnings.\n\nHowever, the    │
│  current economic landscape exhibits a dichotomy; the robust performance of stock indices like the S&P 500      │
│  masks underlying divisions between sectors. While wealthier demographics are seeing continued benefits from    │
│  stock market gains, inflationary pressures threaten to subdue market performance in consumer-focused           │
│  sectors.\n\n## Investor Sentiment  \nInvestors are closely monitoring the Federal Reserve's actions regarding  │
│  interest rates. Many participants are hopeful about potential rate cuts that could stimulate business          │
│  borrowing. However, there remains a cautious outlook due to concerns over how inflationary pressures might     │
│  affect consumer confidence and spending.\n\nThe potential for double-digit corporate earnings growth in 2026   │
│  is both enticing and alarming for investors. While it suggests a strong economic rebound, the shadow of        │
│  inflation creates uncertainty that investors must navigate. Analysts urge a balanced approach to investing to  │
│  align with the changing economic landscape.\n\n## Sector-Specific Outlook  \n1. **Defense and Technology**:    │
│  Growth sectors, notably defense and technology, are benefiting from increased government spending and          │
│  innovations in AI and cybersecurity. Investors see the

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 8e8a3805-1793-420a-96dc-60c57541f28f                                                                     │
│  Agent: Chief Content Officer                                                                                   │
│                                                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 122f472d-7a25-47db-a985-e53298d9104b                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "article": "# Understanding U.S. Inflation and Its Impact on the Stock Market in 2024\n\n## Introduction     │
│  \nAs we delve into 2024, inflation continues to be a significant factor influencing the U.S. economy and       │
│  stock market. After peaking at over 9% in June 2022, the current inflation rate stands at approximately 3.1%.  │
│  This blog post offers insights into how these inflationary trends are impacting investor sentiment, stock      │
│  market performance, and specific sectors.\n\n## Current Inflation Trends  \nInflation remains a pertinent      │
│  issue as the Federal Reserve explores potential interest rate cuts. As of December 2025, inflation in the      │
│  U.S. is approximately 3.1%, reflecting a moderate decline from previous peaks. Despite rising consumer         │
│  prices, the Consumer Price Index (CPI) indicates a more moderate inflationary environment compared to          │
│  previous years. Concerns about tariffs and their subsequent effect on consumer prices linger, contributing to  │
│  the current inflation dynamics.\n\nMoreover, the ongoing fluctuations in tariff policies contribute to         │
│  uncertainties in inflation forecasts. Consumers are becoming increasingly aware of the impact of these         │
│  tariffs, which could stifle purchasing power in the near future. This evolving situation necessitates          │
│  continuous monitoring from both investors and policymakers.\n\n## Impact on the Stock Market  \nThe stock      │
│  market is experiencing notable volatility, reflecting a divergence across various sectors influenced by        │
│  inflation. While sectors like technology and defense are thriving due to increased spending and digital        │
│  advancements, consumer staples are facing challenges from persistent inflation. Notably, the S&P 500 achieved  │
│  all-time highs throughout 2024, buoyed by resilient economic growth and corporate earnings.\n\nHowever, the    │
│  current economic landscape exhibits a dichotomy; the robust performance of stock indices like the S&P 500      │
│  masks underlying divisions between sectors. While wealthier demographics are seeing continued benefits from    │
│  stock market gains, inflationary pressures threaten to subdue market performance in consumer-focused           │
│  sectors.\n\n## Investor Sentiment  \nInvestors are closely monitoring the Federal Reserve's actions regarding  │
│  interest rates. Many participants are hopeful about potential rate cuts that could stimulate business          │
│  borrowing. However, there remains a cautious outlook due to concerns over how inflationary pressures might     │
│  affect consumer confidence and spending.\n\nThe potential for double-digit corporate earnings growth in 2026   │
│  is both enticing and alarming for investors. While it suggests a strong economic rebound, the shadow of        │
│  inflation creates uncertainty that investors must navigate. Analysts urge a balanced approach to investing to  │
│  align with the changing economic landscape.\n\n## Sector-Specific Outlook  \n1. **Defense and Technology**:    │
│  Growth sectors, notably defense and technology, are b

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Social Content

In [8]:
import textwrap

posts = result.pydantic.model_dump()['social_media_posts']
for post in posts:
    platform = post['platform']
    content = post['content']
    print(platform)
    wrapped_content = textwrap.fill(content, width=50)
    print(wrapped_content)
    print('-' * 50)

Twitter
🔍 *2024 Inflation Insights*: Inflation in the U.S.
is currently around 3.1%, a significant drop from
its peak of over 9% in 2022! Explore how these
trends are impacting the stock market.
#InvestSmart #InflationImpact
--------------------------------------------------
LinkedIn
📈 *Market Vibes*: Despite inflation concerns, the
S&P 500 reached new all-time highs in 2024! Which
sectors are thriving? Discover insights into
market dynamics. #StockMarket #2024
--------------------------------------------------
Facebook
🎯 *Investor Strategies*: With inflation pressures
continuing, consider diversifying your portfolio
with TIPS or actively managed funds. What
strategies are you employing? Let’s discuss!
#InvestmentStrategies
--------------------------------------------------
Instagram
📊 Check out our latest blog post for a
comprehensive outlook on inflation’s impact on the
stock market in 2024! Link in bio.
#FinancialLiteracy #MarketTrends
-----------------------------------------------

## Blog Post

In [9]:
from IPython.display import display, Markdown
display(Markdown(result.pydantic.model_dump()['article']))

# Understanding U.S. Inflation and Its Impact on the Stock Market in 2024

## Introduction  
As we delve into 2024, inflation continues to be a significant factor influencing the U.S. economy and stock market. After peaking at over 9% in June 2022, the current inflation rate stands at approximately 3.1%. This blog post offers insights into how these inflationary trends are impacting investor sentiment, stock market performance, and specific sectors.

## Current Inflation Trends  
Inflation remains a pertinent issue as the Federal Reserve explores potential interest rate cuts. As of December 2025, inflation in the U.S. is approximately 3.1%, reflecting a moderate decline from previous peaks. Despite rising consumer prices, the Consumer Price Index (CPI) indicates a more moderate inflationary environment compared to previous years. Concerns about tariffs and their subsequent effect on consumer prices linger, contributing to the current inflation dynamics.

Moreover, the ongoing fluctuations in tariff policies contribute to uncertainties in inflation forecasts. Consumers are becoming increasingly aware of the impact of these tariffs, which could stifle purchasing power in the near future. This evolving situation necessitates continuous monitoring from both investors and policymakers.

## Impact on the Stock Market  
The stock market is experiencing notable volatility, reflecting a divergence across various sectors influenced by inflation. While sectors like technology and defense are thriving due to increased spending and digital advancements, consumer staples are facing challenges from persistent inflation. Notably, the S&P 500 achieved all-time highs throughout 2024, buoyed by resilient economic growth and corporate earnings.

However, the current economic landscape exhibits a dichotomy; the robust performance of stock indices like the S&P 500 masks underlying divisions between sectors. While wealthier demographics are seeing continued benefits from stock market gains, inflationary pressures threaten to subdue market performance in consumer-focused sectors.

## Investor Sentiment  
Investors are closely monitoring the Federal Reserve's actions regarding interest rates. Many participants are hopeful about potential rate cuts that could stimulate business borrowing. However, there remains a cautious outlook due to concerns over how inflationary pressures might affect consumer confidence and spending.

The potential for double-digit corporate earnings growth in 2026 is both enticing and alarming for investors. While it suggests a strong economic rebound, the shadow of inflation creates uncertainty that investors must navigate. Analysts urge a balanced approach to investing to align with the changing economic landscape.

## Sector-Specific Outlook  
1. **Defense and Technology**: Growth sectors, notably defense and technology, are benefiting from increased government spending and innovations in AI and cybersecurity. Investors see these sectors as promising amidst economic fluctuations. 
 
2. **Consumer Staples**: Conversely, industries sensitive to inflation, such as consumer staples, may struggle as price pressures mount, potentially affecting their market performance.

As government budgets expand—particularly under the defense sector—opportunities for growth within high-tech industries appear promising. The sustained demand for innovative solutions continues to attract investor interest across diverse market segments.

## Market Strategies for Investors  
Given this complex economic environment, maintaining a diversified portfolio is crucial for mitigating risks associated with inflation. Investors should consider restructuring their investments based on ongoing market evaluations. 

Strategies to consider include:
- Investing in Treasury Inflation-Protected Securities (TIPS) to safeguard against inflation risks.  
- Opt for actively managed funds that can adapt dynamically to shifting economic conditions, ensuring better resilience during market fluctuations.

*Conclusion*  
The intricate relationship between inflation trends, government policy, and investor sentiment underscores the importance of ongoing vigilance in the financial markets. As we navigate 2024, investors need to be proactive in adjusting their strategies to capitalize on opportunities while being wary of inflation risks. Continuous monitoring of economic indicators will allow for timely adjustments in investment approaches that align with market realities.

---
